# OceanWatch — Entrega 1 · Requisitos 1 y 2

**Requisito 1 — Ingesta documentada:** descarga de los 7 días (1-7 junio 2023) desde la
fuente oficial de NOAA con reintentos y verificación de integridad, descompresión en el
Volume de Unity Catalog y lectura de los CSV con esquema explícito.

**Requisito 2 — Exploración y perfilamiento:** posiciones por día, buques únicos,
distribución por tipo y tamaño, y cuantificación de los problemas de calidad.

Las preguntas de negocio (req. 3) y el almacenamiento óptimo en Delta (req. 4) están en
`02_preguntas_almacenamiento`.

# Requisito 1 — Ingesta

## 1.1 Imports y rutas

In [ ]:
import os
import time
import zipfile

import requests
from pyspark.sql import functions as f

CATALOG = "mine4213"
SCHEMA = "proyecto"
VOLUME = "data"

BASE = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
ZIP_DIR = f"{BASE}/zip"
CSV_DIR = f"{BASE}/csv"

print("BASE:", BASE)
print("ZIP_DIR:", ZIP_DIR)
print("CSV_DIR:", CSV_DIR)

In [ ]:
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}
    COMMENT 'Proyecto OceanWatch Analytics - MINE 4213'
""")

spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}
    COMMENT 'Archivos crudos AIS de NOAA (zip y csv), 1-7 junio 2023'
""")

os.makedirs(ZIP_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)

print("Schema, volumen y directorios listos.")

In [ ]:
URLS = [
    f"https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_{day:02d}.zip"
    for day in range(1, 8)
]

for url in URLS:
    print(url)

## 1.2 Descarga con reintentos y verificación de integridad

Cada zip se valida (existe, no está vacío, es un ZIP válido, CRC correcto de su contenido
y contiene el CSV esperado). Si ya existe y es válido no se vuelve a descargar, de modo
que el notebook se puede re-ejecutar sin bajar de nuevo ~2,3 GB.

In [ ]:
def validate_zip(zip_path, expected_csv_name):
    """
    Valida la integridad básica de un archivo ZIP.

    Comprueba:
    - existencia
    - tamaño > 0
    - estructura ZIP válida
    - CRC de los archivos internos
    - presencia del CSV esperado
    """

    if not os.path.exists(zip_path):
        return False, "El archivo no existe"

    if os.path.getsize(zip_path) == 0:
        return False, "El archivo está vacío"

    if not zipfile.is_zipfile(zip_path):
        return False, "El archivo no es un ZIP válido"

    try:
        with zipfile.ZipFile(zip_path, "r") as z:
            bad_file = z.testzip()

            if bad_file is not None:
                return False, f"Error CRC en {bad_file}"

            files = z.namelist()

            if expected_csv_name not in files:
                return False, f"No contiene {expected_csv_name}"

    except Exception as e:
        return False, str(e)

    return True, "OK"

In [ ]:
def download_with_retries(url, destination, max_retries=3, timeout=120):
    """
    Descarga un archivo con reintentos y backoff lineal.
    """

    for attempt in range(1, max_retries + 1):

        try:
            print(f"Intento {attempt}/{max_retries}: {url}")

            with requests.get(
                url,
                stream=True,
                timeout=timeout
            ) as response:

                response.raise_for_status()

                with open(destination, "wb") as file:

                    for chunk in response.iter_content(
                        chunk_size=1024 * 1024
                    ):
                        if chunk:
                            file.write(chunk)

            print(
                f"Descargado: {os.path.basename(destination)} "
                f"({os.path.getsize(destination) / 1024**2:.2f} MB)"
            )

            return

        except Exception as e:

            print(f"Error: {e}")

            if attempt == max_retries:
                raise

            wait_seconds = attempt * 5

            print(f"Reintentando en {wait_seconds} segundos...")
            time.sleep(wait_seconds)

In [ ]:
download_results = []

for day, url in enumerate(URLS, start=1):

    zip_name = f"AIS_2023_06_{day:02d}.zip"
    csv_name = f"AIS_2023_06_{day:02d}.csv"

    zip_path = f"{ZIP_DIR}/{zip_name}"

    valid, message = validate_zip(
        zip_path,
        csv_name
    )

    if valid:
        print(f"{zip_name}: ya existe y es válido.")

    else:
        print(f"{zip_name}: debe descargarse. Motivo: {message}")

        download_with_retries(
            url,
            zip_path
        )

        valid, message = validate_zip(
            zip_path,
            csv_name
        )

        if not valid:
            raise RuntimeError(
                f"Falló la verificación de {zip_name}: {message}"
            )

    download_results.append(
        (
            zip_name,
            os.path.getsize(zip_path),
            valid,
            message
        )
    )

In [ ]:
download_df = spark.createDataFrame(
    download_results,
    [
        "archivo",
        "bytes",
        "integridad_ok",
        "resultado_validacion"
    ]
)

display(download_df)

## 1.3 Descompresión en el Volume

Spark no lee zip directamente, así que cada CSV se extrae al directorio `csv/` del Volume.

In [ ]:
for day in range(1, 8):

    zip_name = f"AIS_2023_06_{day:02d}.zip"
    csv_name = f"AIS_2023_06_{day:02d}.csv"

    zip_path = f"{ZIP_DIR}/{zip_name}"
    csv_path = f"{CSV_DIR}/{csv_name}"

    if os.path.exists(csv_path) and os.path.getsize(csv_path) > 0:
        print(f"{csv_name}: ya existe.")

    else:
        print(f"Extrayendo {zip_name}...")

        with zipfile.ZipFile(zip_path, "r") as z:
            z.extract(csv_name, CSV_DIR)

        print(f"Extraído: {csv_name}")

In [ ]:
csv_files = sorted([
    name
    for name in os.listdir(CSV_DIR)
    if name.startswith("AIS_2023_06_") and name.endswith(".csv")
])

expected_files = {
    f"AIS_2023_06_{day:02d}.csv"
    for day in range(1, 8)
}

actual_files = set(csv_files)

missing_files = expected_files - actual_files
unexpected_files = actual_files - expected_files

print(f"Cantidad de CSV encontrados: {len(csv_files)}")
print("Faltantes:", missing_files)
print("No esperados:", unexpected_files)

assert len(missing_files) == 0, (
    f"Faltan archivos: {missing_files}"
)

assert len(actual_files) == 7, (
    f"Se esperaban 7 archivos y hay {len(actual_files)}"
)

## 1.4 Esquema

Se verifica que los 7 archivos compartan el mismo encabezado antes de fijar el esquema.

In [ ]:
headers = {}

for file_name in csv_files:

    with open(f"{CSV_DIR}/{file_name}", "r", encoding="utf-8") as file:
        headers[file_name] = file.readline().strip()

unique_headers = set(headers.values())

print("Cantidad de encabezados diferentes:", len(unique_headers))

for header in unique_headers:
    print(header)

assert len(unique_headers) == 1, (
    "Los archivos no comparten el mismo encabezado"
)

## 1.5 Lectura con esquema explícito

El esquema explícito evita el `inferSchema` (una pasada extra completa sobre ~6 GB de CSV)
y fija los tipos: `MMSI` como string (es un identificador, no un número), `BaseDateTime`
como timestamp y los códigos AIS como enteros cortos.

In [ ]:
AIS_SCHEMA = """
    MMSI string,
    BaseDateTime timestamp,
    LAT double,
    LON double,
    SOG float,
    COG float,
    Heading float,
    VesselName string,
    IMO string,
    CallSign string,
    VesselType smallint,
    Status smallint,
    Length float,
    Width float,
    Draft float,
    Cargo string,
    TransceiverClass string
"""

ais = (
    spark.read
    .option("header", True)
    .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
    .schema(AIS_SCHEMA)
    .csv(f"{CSV_DIR}/AIS_2023_06_*.csv")
    .withColumn("fecha", f.to_date("BaseDateTime"))
    .withColumn("archivo_origen", f.col("_metadata.file_name"))
    .withColumn(
        "fecha_archivo",
        f.to_date(
            f.regexp_extract(
                f.col("_metadata.file_name"),
                r"AIS_(\d{4}_\d{2}_\d{2})",
                1
            ),
            "yyyy_MM_dd"
        )
    )
)

ais.printSchema()

display(ais.limit(20))

## 1.6 Validaciones de la lectura

In [ ]:
archivos_leidos = sorted(
    row["archivo_origen"]
    for row in ais.select("archivo_origen").distinct().collect()
)

print(f"Spark leyó {len(archivos_leidos)} archivos:")

for name in archivos_leidos:
    print(name)

assert len(archivos_leidos) == 7, (
    f"Spark debería leer 7 archivos, pero leyó {len(archivos_leidos)}."
)

In [ ]:
expected_dates = {
    f"2023-06-{day:02d}"
    for day in range(1, 8)
}

actual_dates = {
    str(row["fecha"])
    for row in ais.select("fecha").distinct().collect()
}

print("Fechas encontradas:", sorted(actual_dates))

assert actual_dates == expected_dates

# Requisito 2 — Exploración y perfilamiento

## 2.1 Posiciones y buques únicos por día

In [ ]:
perfil_diario = (
    ais
    .groupBy("fecha")
    .agg(
        f.count("*").alias("posiciones"),
        f.countDistinct("MMSI").alias("buques_unicos")
    )
    .orderBy("fecha")
)

display(perfil_diario)

### Total semanal

In [ ]:
resumen_general = (
    ais
    .agg(
        f.count("*").alias("total_posiciones"),
        f.countDistinct("MMSI").alias("buques_unicos_semana"),
        f.min("BaseDateTime").alias("primer_timestamp"),
        f.max("BaseDateTime").alias("ultimo_timestamp")
    )
)

display(resumen_general)

## 2.2 Completitud

In [ ]:
columnas_originales = [
    "MMSI",
    "BaseDateTime",
    "LAT",
    "LON",
    "SOG",
    "COG",
    "Heading",
    "VesselName",
    "IMO",
    "CallSign",
    "VesselType",
    "Status",
    "Length",
    "Width",
    "Draft",
    "Cargo",
    "TransceiverClass"
]

string_cols = {
    field.name
    for field in ais.schema.fields
    if field.dataType.simpleString() == "string"
}

missing_exprs = []

for col_name in columnas_originales:

    if col_name in string_cols:
        missing_condition = (
            f.col(col_name).isNull()
            | (f.trim(f.col(col_name)) == "")
        )
    else:
        missing_condition = f.col(col_name).isNull()

    missing_exprs.append(
        f.sum(
            f.when(missing_condition, 1).otherwise(0)
        ).alias(col_name)
    )

missing_row = (
    ais
    .agg(
        f.count("*").alias("total_filas"),
        *missing_exprs
    )
    .first()
)

total_filas = missing_row["total_filas"]

perfil_completitud = spark.createDataFrame(
    [
        (
            col_name,
            int(missing_row[col_name]),
            round(
                100 * missing_row[col_name] / total_filas,
                4
            )
        )
        for col_name in columnas_originales
    ],
    [
        "columna",
        "valores_faltantes",
        "porcentaje_faltante"
    ]
)

display(
    perfil_completitud
    .orderBy(f.desc("porcentaje_faltante"))
)

## 2.3 Distribución por tipo de buque

In [ ]:
distribucion_tipo = (
    ais
    .groupBy("VesselType")
    .agg(
        f.count("*").alias("posiciones"),
        f.countDistinct("MMSI").alias("buques_unicos")
    )
    .withColumn(
        "porcentaje_posiciones",
        f.round(
            100
            * f.col("posiciones")
            / f.lit(total_filas),
            4
        )
    )
    .orderBy(f.desc("posiciones"))
)

display(distribucion_tipo)

In [ ]:
display(
    distribucion_tipo
    .filter(f.col("VesselType").isNull())
)

In [ ]:
display(distribucion_tipo.limit(20))

## 2.4 Distribución por tamaño

In [ ]:
dimensiones_validas = (
    ais
    .select(
        f.when(f.col("Length") > 0, f.col("Length")).alias("Length"),
        f.when(f.col("Width") > 0, f.col("Width")).alias("Width"),
        f.when(f.col("Draft") > 0, f.col("Draft")).alias("Draft")
    )
)

display(
    dimensiones_validas.summary(
        "count",
        "mean",
        "stddev",
        "min",
        "25%",
        "50%",
        "75%",
        "90%",
        "95%",
        "99%",
        "max"
    )
)

### Revisión dimensiones extremas

In [ ]:
columnas_dimensiones = [
    "MMSI",
    "VesselName",
    "VesselType",
    "Length",
    "Width",
    "Draft"
]

buques_dimensiones = (
    ais
    .select(*columnas_dimensiones)
    .dropDuplicates(columnas_dimensiones)
)

display(
    buques_dimensiones
    .orderBy(f.desc("Length"))
    .limit(50)
)

In [ ]:
display(
    buques_dimensiones
    .orderBy(f.desc("Width"))
    .limit(50)
)

In [ ]:
ais_tamano = (
    ais
    .withColumn(
        "categoria_tamano",
        f.when(
            f.col("Length").isNull() | (f.col("Length") <= 0),
            "Sin longitud utilizable"
        )
        .when(f.col("Length") < 25, "< 25 m")
        .when(f.col("Length") < 50, "25 - 49.9 m")
        .when(f.col("Length") < 100, "50 - 99.9 m")
        .when(f.col("Length") < 200, "100 - 199.9 m")
        .otherwise(">= 200 m")
    )
)

distribucion_tamano = (
    ais_tamano
    .groupBy("categoria_tamano")
    .agg(
        f.count("*").alias("posiciones"),
        f.countDistinct("MMSI").alias("buques_unicos")
    )
    .withColumn(
        "porcentaje_posiciones",
        f.round(
            100
            * f.col("posiciones")
            / f.lit(total_filas),
            4
        )
    )
    .orderBy(f.desc("posiciones"))
)

display(distribucion_tamano)

In [ ]:
tipo_tamano = (
    ais_tamano
    .groupBy(
        "VesselType",
        "categoria_tamano"
    )
    .agg(
        f.count("*").alias("posiciones"),
        f.countDistinct("MMSI").alias("buques_unicos")
    )
    .orderBy(f.desc("posiciones"))
)

display(tipo_tamano.limit(50))

## 2.5 Reglas de calidad

In [ ]:
inicio_semana = f.lit("2023-06-01 00:00:00").cast("timestamp")
fin_semana = f.lit("2023-06-08 00:00:00").cast("timestamp")

sog_no_disponible = (
    f.abs(f.col("SOG") - f.lit(102.3)) < 0.001
)

checks = [
    (
        "coordenadas_nulas",
        f.col("LAT").isNull() | f.col("LON").isNull(),
        "LAT o LON no disponibles"
    ),
    (
        "lat_fuera_rango",
        (f.col("LAT") < -90) | (f.col("LAT") > 90),
        "Latitud fuera de [-90, 90]"
    ),
    (
        "lon_fuera_rango",
        (f.col("LON") < -180) | (f.col("LON") > 180),
        "Longitud fuera de [-180, 180]"
    ),

    (
        "sog_nulo",
        f.col("SOG").isNull(),
        "Velocidad no informada"
    ),
    (
        "sog_no_disponible_102_3",
        sog_no_disponible,
        "102.3 corresponde al valor AIS de SOG no disponible"
    ),
    (
        "sog_fuera_codificacion_ais",
        (f.col("SOG") < 0)
        | (
            (f.col("SOG") > 102.3)
            & (~sog_no_disponible)
        ),
        "Valor fuera del rango esperado para SOG AIS"
    ),
    (
        "sog_mayor_60_sospechosa",
        (f.col("SOG") > 60) & (f.col("SOG") <= 102.2),
        "Velocidad muy alta: revisar, no eliminar automáticamente"
    ),

    (
        "cog_no_disponible_360",
        f.abs(f.col("COG") - f.lit(360.0)) < 0.001,
        "COG=360 significa no disponible"
    ),
    (
        "cog_fuera_rango",
        (f.col("COG") < 0) | (f.col("COG") > 360),
        "COG fuera del rango AIS"
    ),

    (
        "heading_no_disponible_511",
        f.abs(f.col("Heading") - f.lit(511.0)) < 0.001,
        "Heading=511 significa no disponible"
    ),
    (
        "heading_fuera_rango",
        (f.col("Heading") < 0)
        | (
            (f.col("Heading") > 359)
            & (f.abs(f.col("Heading") - f.lit(511.0)) >= 0.001)
        ),
        "Heading distinto de 0-359 y no es el sentinel 511"
    ),

    (
        "mmsi_nulo_o_vacio",
        f.col("MMSI").isNull()
        | (f.trim(f.col("MMSI")) == ""),
        "MMSI ausente"
    ),
    (
        "mmsi_formato_invalido",
        f.col("MMSI").isNotNull()
        & (~f.trim(f.col("MMSI")).rlike(r"^[0-9]{9}$")),
        "MMSI que no contiene exactamente 9 dígitos"
    ),

    (
        "timestamp_nulo",
        f.col("BaseDateTime").isNull(),
        "Timestamp ausente"
    ),
    (
        "timestamp_fuera_semana",
        (f.col("BaseDateTime") < inicio_semana)
        | (f.col("BaseDateTime") >= fin_semana),
        "Timestamp fuera del corpus 1-7 junio"
    ),
    (
        "fecha_archivo_no_extraida",
        f.col("fecha_archivo").isNull(),
        "No se pudo extraer la fecha del nombre del archivo de origen"
    ),
    (
        "fecha_no_coincide_archivo",
        f.col("fecha").isNotNull()
        & f.col("fecha_archivo").isNotNull()
        & (f.col("fecha") != f.col("fecha_archivo")),
        "El timestamp no corresponde al día indicado por el archivo"
    ),

    (
        "length_negativo",
        f.col("Length") < 0,
        "Longitud físicamente inválida"
    ),
    (
        "width_negativo",
        f.col("Width") < 0,
        "Ancho físicamente inválido"
    ),
    (
        "draft_negativo",
        f.col("Draft") < 0,
        "Calado físicamente inválido"
    ),

    (
        "length_cero",
        f.col("Length") == 0,
        "Longitud cero; posiblemente no disponible"
    ),
    (
        "width_cero",
        f.col("Width") == 0,
        "Ancho cero; posiblemente no disponible"
    ),
    (
        "draft_cero",
        f.col("Draft") == 0,
        "Calado cero; posiblemente no disponible"
    ),

    (
        "vessel_type_nulo",
        f.col("VesselType").isNull(),
        "Tipo de buque no informado"
    )
]

In [ ]:
quality_exprs = []

for i, (_, condition, _) in enumerate(checks):
    quality_exprs.append(
        f.sum(
            f.when(condition, 1).otherwise(0)
        ).alias(f"q_{i}")
    )

quality_row = (
    ais
    .agg(
        f.count("*").alias("total_filas"),
        *quality_exprs
    )
    .first()
)

total_filas = quality_row["total_filas"]

In [ ]:
quality_data = []

for i, (nombre, _, descripcion) in enumerate(checks):

    cantidad = int(quality_row[f"q_{i}"])

    porcentaje = round(
        100 * cantidad / total_filas,
        6
    )

    quality_data.append(
        (
            nombre,
            cantidad,
            porcentaje,
            descripcion
        )
    )

diagnostico_calidad = spark.createDataFrame(
    quality_data,
    [
        "regla",
        "filas_afectadas",
        "porcentaje",
        "interpretacion"
    ]
)

display(
    diagnostico_calidad
    .orderBy(f.desc("filas_afectadas"))
)

### Headings y COG anómalos

In [ ]:
heading_anomalos = (
    ais
    .filter(
        (f.col("Heading") < 0)
        | (
            (f.col("Heading") > 359)
            & (f.abs(f.col("Heading") - 511.0) >= 0.001)
        )
    )
    .groupBy("Heading")
    .agg(
        f.count("*").alias("apariciones")
    )
    .orderBy(
        f.desc("apariciones")
    )
)

display(heading_anomalos)

In [ ]:
cog_anomalos = (
    ais
    .filter(
        (f.col("COG") < 0)
        | (f.col("COG") > 360)
    )
    .groupBy("COG")
    .agg(
        f.count("*").alias("apariciones")
    )
    .orderBy(
        f.desc("apariciones")
    )
)

display(cog_anomalos)

## 2.6 Velocidades altas

In [ ]:
display(
    ais
    .filter(
        (f.col("SOG") > 60)
        & (f.col("SOG") <= 102.2)
    )
    .select(
        "MMSI",
        "BaseDateTime",
        "LAT",
        "LON",
        "SOG",
        "VesselType",
        "VesselName"
    )
    .orderBy(f.desc("SOG"))
    .limit(100)
)

In [ ]:
display(
    ais
    .select("SOG")
    .filter(f.col("SOG").isNotNull())
    .summary(
        "count",
        "mean",
        "stddev",
        "min",
        "50%",
        "90%",
        "95%",
        "99%",
        "max"
    )
)

## 2.7 Duplicados

Duplicado = mismo MMSI + BaseDateTime

In [ ]:
duplicados_timestamp = (
    ais
    .filter(
        f.col("MMSI").isNotNull()
        & f.col("BaseDateTime").isNotNull()
    )
    .groupBy(
        "MMSI",
        "BaseDateTime"
    )
    .agg(
        f.count("*").alias("numero_registros"),
        f.countDistinct(
            f.struct("LAT", "LON")
        ).alias("posiciones_distintas")
    )
    .filter(
        f.col("numero_registros") > 1
    )
)

In [ ]:
resumen_duplicados = (
    duplicados_timestamp
    .agg(
        f.count("*").alias(
            "claves_mmsi_timestamp_duplicadas"
        ),

        f.sum("numero_registros").alias(
            "filas_en_claves_duplicadas"
        ),

        f.sum(
            f.col("numero_registros") - 1
        ).alias(
            "filas_excedentes"
        ),

        f.sum(
            f.when(
                f.col("posiciones_distintas") == 1,
                1
            ).otherwise(0)
        ).alias(
            "claves_con_misma_posicion"
        ),

        f.sum(
            f.when(
                f.col("posiciones_distintas") > 1,
                1
            ).otherwise(0)
        ).alias(
            "claves_con_posiciones_conflictivas"
        )
    )
)

display(resumen_duplicados)

In [ ]:
display(
    duplicados_timestamp
    .orderBy(
        f.desc("numero_registros")
    )
    .limit(100)
)

In [ ]:
display(
    duplicados_timestamp
    .filter(
        f.col("posiciones_distintas") > 1
    )
    .orderBy(
        f.desc("numero_registros")
    )
    .limit(100)
)

### Duplicados por contenido completo

In [ ]:
ais_hash = (
    ais
    .withColumn(
        "_row_hash",
        f.xxhash64(
            *[f.col(c) for c in columnas_originales]
        )
    )
)

duplicados_fila = (
    ais_hash
    .groupBy("_row_hash")
    .agg(
        f.count("*").alias("numero_registros")
    )
    .filter(
        f.col("numero_registros") > 1
    )
)

In [ ]:
resumen_duplicados_fila = (
    duplicados_fila
    .agg(
        f.count("*").alias(
            "grupos_repetidos"
        ),
        f.sum("numero_registros").alias(
            "filas_en_grupos_repetidos"
        ),
        f.sum(
            f.col("numero_registros") - 1
        ).alias(
            "filas_excedentes"
        )
    )
)

display(resumen_duplicados_fila)

## 2.8 MMSI anómalos

In [ ]:
mmsi_anomalos = (
    ais
    .filter(
        f.col("MMSI").isNull()
        | (f.trim(f.col("MMSI")) == "")
        | (~f.trim(f.col("MMSI")).rlike(r"^[0-9]{9}$"))
    )
    .groupBy("MMSI")
    .agg(
        f.count("*").alias("posiciones")
    )
    .orderBy(
        f.desc("posiciones")
    )
)

display(mmsi_anomalos)

## 2.9 Coordenadas anómalas

In [ ]:
display(
    ais
    .filter(
        (f.col("LAT") < -90)
        | (f.col("LAT") > 90)
        | (f.col("LON") < -180)
        | (f.col("LON") > 180)
    )
    .groupBy(
        "LAT",
        "LON"
    )
    .agg(
        f.count("*").alias("apariciones")
    )
    .orderBy(
        f.desc("apariciones")
    )
)

## 2.10 Problemas de calidad por día

In [ ]:
calidad_por_dia = (
    ais
    .groupBy("fecha")
    .agg(
        f.count("*").alias("posiciones"),

        f.sum(
            f.when(
                (f.col("LAT") < -90)
                | (f.col("LAT") > 90)
                | (f.col("LON") < -180)
                | (f.col("LON") > 180),
                1
            ).otherwise(0)
        ).alias("coordenadas_fuera_rango"),

        f.sum(
            f.when(
                f.abs(f.col("SOG") - 102.3) < 0.001,
                1
            ).otherwise(0)
        ).alias("sog_no_disponible"),

        f.sum(
            f.when(
                f.col("MMSI").isNull()
                | (~f.trim(f.col("MMSI")).rlike(r"^[0-9]{9}$")),
                1
            ).otherwise(0)
        ).alias("mmsi_anomalo")
    )
    .orderBy("fecha")
)

display(calidad_por_dia)

## 2.11 Conclusiones del perfilamiento

El corpus analizado contiene **60.533.559 posiciones AIS** correspondientes a
**31.871 MMSI distintos**, registradas entre el 1 y el 7 de junio de 2023.
El volumen diario se mantiene entre aproximadamente 8,0 y 9,1 millones de
posiciones, sin observarse días ausentes dentro del periodo analizado.

### Completitud

Las variables fundamentales para el análisis de posición presentan una alta
completitud. MMSI, BaseDateTime, LAT, LON, SOG, COG, Heading y
TransceiverClass no contienen valores nulos.

La principal limitación de completitud se encuentra en atributos estáticos de
las embarcaciones:

- Draft: 64,43% de valores faltantes.
- IMO: 42,77%.
- Status: 32,96%.
- Cargo: 32,87%.
- CallSign: 16,78%.
- Width: 15,23%.
- Length: 6,05%.

Adicionalmente, algunos atributos dimensionales utilizan el valor cero, que no
resulta útil como dimensión física. Considerando tanto nulos como valores cero,
aproximadamente el 9,49% de las posiciones no cuenta con una longitud
utilizable, el 18,91% no cuenta con ancho utilizable y el 68,92% no cuenta con
calado utilizable.

### Validez

No se encontraron coordenadas fuera de los rangos globales válidos de latitud
y longitud, posiciones con coordenadas nulas, timestamps fuera del periodo
analizado ni inconsistencias entre la fecha del mensaje y el archivo de origen.

Tampoco se encontraron valores SOG fuera de la codificación AIS utilizada en
el dataset.

Se identificaron, sin embargo, algunos valores anómalos:

- 49.897 posiciones (0,0824%) presentan MMSI que no cumplen el formato
  esperado de nueve dígitos.
- 1.061 posiciones (0,00175%) presentan Heading fuera del rango esperado y
  diferente del valor sentinel 511.
- 12 posiciones presentan COG superior al rango esperado.
- 626 posiciones presentan velocidades superiores a 60 nudos y se consideran
  sospechosas para revisión, aunque no se clasifican automáticamente como
  inválidas.

### Valores no disponibles definidos por AIS

Se encontraron valores especiales del estándar que representan información no
disponible y, por lo tanto, no deben confundirse con errores de calidad:

- Heading = 511: 33.535.628 posiciones (55,40%).
- COG = 360: 10.291.131 posiciones (17,00%).
- SOG = 102.3: 159.987 posiciones (0,264%).

Estos valores deberán ser tratados de manera diferenciada en etapas posteriores
de limpieza y transformación.

### Duplicados

Se identificaron **1.672 combinaciones MMSI + BaseDateTime repetidas**, que
involucran 3.344 registros y representan 1.672 filas excedentes.

De estas claves repetidas:

- 1.400 presentan la misma posición.
- 272 presentan posiciones distintas para el mismo MMSI y timestamp, lo que
  constituye una inconsistencia que requiere tratamiento específico.

Al comparar el contenido completo de los registros mediante un hash de fila se
encontraron **1.388 grupos repetidos**, correspondientes a 2.776 filas y
1.388 filas excedentes por contenido.

Esto indica que no todos los registros que comparten MMSI, timestamp y posición
son duplicados completos; algunos presentan diferencias en otros atributos.

### Conclusión

El dataset presenta una calidad adecuada para el análisis de tráfico marítimo
en sus variables principales de posición y tiempo, pero contiene problemas
relevantes en los atributos estáticos de los buques, identificadores MMSI,
valores especiales de navegación y duplicados.

El diagnóstico realizado en esta etapa servirá como base para definir las
reglas de limpieza, normalización y control de calidad de la siguiente fase
del proyecto.